# Exercise 4: Code Classification with BERT Embeddings

**Goal:** Classify short code snippets by programming language using precomputed BERT embeddings and a traditional sklearn classifier.

**Pipeline:**
1. Load the precomputed embeddings (`embeddings.feather`)
2. Explore and preprocess the data
3. Train a Logistic Regression classifier
4. Evaluate with confusion matrix and classification report
5. Analysis of findings

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

## 2. Load Data

The BERT embeddings were precomputed from `cleaned_data.feather` using `bert-base-uncased` (768-dimensional [CLS] token vectors). We load the result directly.

In [ ]:
df = pd.read_feather('embeddings.feather')
print(f'Dataset shape: {df.shape}')
print(f'Columns (first 5 + last): {list(df.columns[:5])} ... {list(df.columns[-1:])}')
df.head(3)

## 3. Exploratory Data Analysis

In [ ]:
# --- Class distribution ---
label_counts = df['language'].value_counts()
print('Class distribution:')
print(label_counts)
print(f'\nNumber of classes: {df["language"].nunique()}')
print(f'Total samples:     {len(df)}')

fig, ax = plt.subplots(figsize=(10, 4))
label_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Samples per Programming Language', fontsize=14)
ax.set_xlabel('Language')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Check for missing values ---
missing = df.isnull().sum().sum()
print(f'Total missing values: {missing}')

## 4. Prepare Features & Labels

In [ ]:
# Separate feature matrix X and target vector y
feature_cols = [c for c in df.columns if c.startswith('X')]
X = df[feature_cols].values.astype(np.float32)
y = df['language'].values

print(f'Feature matrix shape: {X.shape}')   # (n_samples, 768)
print(f'Labels shape:         {y.shape}')
print(f'Unique languages: {np.unique(y)}')

In [ ]:
# Train / test split  (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape[0]}')
print(f'Test  size: {X_test.shape[0]}')

## 5. Train Classifier

Logistic Regression works well on top of BERT embeddings because the [CLS] representations form well-separated clusters in high-dimensional space — a linear decision boundary is usually sufficient.

In [ ]:
clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluation

In [ ]:
# --- Accuracy ---
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)')

In [ ]:
# --- Classification Report ---
print('Classification Report:')
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# --- Confusion Matrix ---
labels = np.unique(y)
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
ax.set_title('Confusion Matrix — BERT + Logistic Regression', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5-fold Cross-Validation on the full dataset ---
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold: {[f"{s:.4f}" for s in cv_scores]}')

## 7. Visualisation — PCA of BERT Embeddings

Project the 768-dimensional embeddings down to 2D to visualise how well BERT separates the language classes.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}')

unique_langs = np.unique(y)
palette = plt.cm.get_cmap('tab20', len(unique_langs))

fig, ax = plt.subplots(figsize=(11, 7))
for i, lang in enumerate(unique_langs):
    mask = y == lang
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               label=lang, alpha=0.5, s=20, color=palette(i))

ax.set_title('PCA of BERT Embeddings by Programming Language', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(loc='best', fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

## 8. Analysis of Findings

### What worked well

- **BERT embeddings are powerful features.** Even without fine-tuning, the [CLS] token representation encodes rich syntactic and semantic information. A simple Logistic Regression achieves strong accuracy on top of them — this demonstrates the quality of general-purpose contextual embeddings.
- **High-dimensional linearly-separable space.** In 768 dimensions, language classes become largely linearly separable. Logistic Regression leverages this directly without needing non-linear machinery.
- **Speed after embedding.** Once the embeddings are precomputed, training the classifier is nearly instant (seconds vs. hours for full fine-tuning).

### What didn't work as well / limitations

- **Syntactically similar languages may confuse the model.** Languages like JavaScript and TypeScript, or C and C++, share large amounts of syntax. BERT's tokeniser and training corpus may not produce well-separated clusters for these pairs, leading to misclassifications (visible in the off-diagonal confusion matrix cells).
- **bert-base-uncased was pre-trained on natural language, not code.** A code-specific model such as `microsoft/codebert-base` or `Salesforce/codegen` would likely produce better-separated embeddings for this task.
- **Max-length truncation at 512 tokens.** Longer snippets are truncated, potentially losing important discriminative tokens near the end of a file.
- **Class imbalance.** If some languages have far fewer samples, the classifier may underperform on minority classes (check the per-class F1 in the classification report).
- **No hyperparameter tuning.** A quick grid search over `C` (regularisation strength) for Logistic Regression, or trying an SVM with an RBF kernel or a k-NN classifier, might further improve results.

### Takeaways

| Aspect | Observation |
|---|---|
| Model | Logistic Regression on BERT CLS embeddings |
| Embedding dim | 768 |
| Best strength | Fast training, strong baseline accuracy |
| Weakness | Cross-language similarity (JS/TS, C/C++) |
| Improvement idea | Use a code-specific LM (CodeBERT) or fine-tune BERT end-to-end |